In [84]:
import pandas as pd
CURRENT_SEASON="Spring2026"

In [85]:
df = pd.read_csv('../data/raw/softball_stats_raw.csv')

In [86]:
active = pd.read_csv('../data/roster/active_roster.csv')

In [87]:
active.loc[active['IsActive'] == 1, "PlayerName"]

0     Nathaniel Lastrapes
1            Beto Becerra
2              David Cruz
3            Nick Llorens
5             Jon Navarro
6         Tony Plascencia
7            Joab Santoyo
8           Angel Santoyo
9           Carlos Aranda
10              Jay Munoz
11             Noe Romero
19          Mike Quintero
20             Angel Cruz
Name: PlayerName, dtype: str

In [102]:
def filter_active_current_season(stats_df, roster_df, season=CURRENT_SEASON):
    season_stats = stats_df[stats_df['Season'] == season]
    active_players = roster_df.loc[roster_df['IsActive'] == 1, "PlayerName"]
    return season_stats[season_stats['PlayerName'].isin(active_players)]

def aggregate_stats(stats_df):
    stats_df = stats_df.assign(
        SeasonGame=stats_df["Season"].astype(str) + "_" + stats_df["Game"].astype(str)
    )
    
    agg = stats_df.groupby("PlayerName").agg(
        Games_Played=("SeasonGame", "nunique"),
        AB=("AB", "sum"),
        H=("H", "sum"),
        BB=("BB", "sum"),
        SF=("SF", "sum")
    )

    agg['AVG'] = (agg['H'] / agg['AB']).round(3)
    agg['OBP'] = ((agg["H"] + agg["BB"]) / (agg["AB"] + agg["BB"] + agg["SF"])).round(3)

    agg = agg.sort_values(by=['AVG', 'OBP', 'AB'], ascending=False)
    return agg.reset_index()

def season_by_season_stats(stats_df):
    stats_df = stats_df.assign(
        SeasonGame=stats_df["Season"].astype(str) + "_" + stats_df["Game"].astype(str)
    )
    
    agg = stats_df.groupby(["PlayerName", "Season"]).agg(
        Games_Played=("SeasonGame", "nunique"),
        AB=("AB", "sum"),
        H=("H", "sum"),
        BB=("BB", "sum"),
        SF=("SF", "sum")
    )

    agg['AVG'] = (agg['H'] / agg['AB']).round(3)
    agg['OBP'] = ((agg["H"] + agg["BB"]) / (agg["AB"] + agg["BB"] + agg["SF"])).round(3)

    agg = agg.sort_values(by=['AVG', 'OBP', 'AB'], ascending=False)
    return agg.reset_index()

In [89]:
cs = filter_active_current_season(df, active)

aggregate_stats(cs)

,PlayerName,Games_Played,AB,H,BB,SF,AVG,OBP
0,Noe Romero,2,5,5,0,0,1.000,1.000
1,Joab Santoyo,3,11,10,0,0,0.909,0.909
2,Nick Llorens,3,11,9,0,1,0.818,0.750
3,Angel Santoyo,2,4,3,1,0,0.750,0.800
4,Tony Plascencia,4,15,10,0,0,0.667,0.667
5,Beto Becerra,4,14,9,1,0,0.643,0.667
6,Angel Cruz,4,14,9,0,0,0.643,0.643
7,Mike Quintero,4,14,7,0,1,0.500,0.467
8,Carlos Aranda,3,11,5,0,0,0.455,0.455
9,Nathaniel Lastrapes,3,11,5,0,0,0.455,0.455


In [90]:
def filter_active_player_stats(stats_df, roster_df):
    active_players = roster_df.loc[roster_df['IsActive'] == 1, "PlayerName"]
    return stats_df[stats_df['PlayerName'].isin(active_players)]

In [91]:
filter_active_player_stats(df, active)

,Season,Game,PlayerName,AB,H,BB,SF
0,Winter2023,1,Nathaniel Lastrapes,2,2,1,0
1,Winter2023,1,Beto Becerra,1,0,2,0
2,Winter2023,1,David Cruz,2,1,0,0
7,Winter2023,1,Jon Navarro,2,1,0,0
8,Winter2023,1,Tony Plascencia,3,0,0,0
...,...,...,...,...,...,...,...
1103,Spring2026,4,Nick Llorens,2,2,0,1
1104,Spring2026,4,Noe Romero,1,1,0,0
1105,Spring2026,4,Carlos Aranda,3,1,0,0
1106,Spring2026,4,David Cruz,3,0,0,0


In [92]:
LAST_SEASON = 'Winter2026'
def build_batting_order(stats_df, roster_df):
    active_players = active.loc[active['IsActive'] == 1, "PlayerName"]

    last_season_df = df[
        (df['Season'] == LAST_SEASON)
        & (df["PlayerName"].isin(active_players))]

    vets = aggregate_stats(last_season_df).sort_values(['AVG', 'OBP'], ascending=False)

    vet_names = set(vets["PlayerName"])
    new_names = [name for name in active_players if name not in vet_names]
    new_players = pd.DataFrame({"PlayerName": new_names})

    order = pd.concat([vets, new_players], ignore_index=True)
    return order[['PlayerName']]

In [93]:
build_batting_order(df, active)

,PlayerName
0,Joab Santoyo
1,Nick Llorens
2,Noe Romero
3,Angel Santoyo
4,Carlos Aranda
5,David Cruz
6,Tony Plascencia
7,Beto Becerra
8,Jay Munoz
9,Nathaniel Lastrapes


In [ ]:
active_players = active.loc[active['IsActive'] == 1, "PlayerName"]

last_season_df = df[
    (df['Season'] == LAST_SEASON)
    & (df["PlayerName"].isin(active_players))]

vets = aggregate_stats(last_season_df).sort_values(['AVG', 'OBP'], ascending=False)

vet_names = set(vets["PlayerName"])
new_names = [name for name in active_players if name not in vet_names]
new_players = pd.DataFrame({"PlayerName": new_names})

order = pd.concat([vets, new_players], ignore_index=True)
return order[['PlayerName']]


,PlayerName
0,Joab Santoyo
1,Nick Llorens
2,Noe Romero
3,Angel Santoyo
4,Carlos Aranda
5,David Cruz
6,Tony Plascencia
7,Beto Becerra
8,Jay Munoz
9,Nathaniel Lastrapes


In [ ]:

season_by_season_stats(df)

,PlayerName,Season,Games_Played,AB,H,BB,SF,AVG,OBP
0,Noe Romero,Spring2026,2,5,5,0,0,1.000,1.000
1,Noe Romero,Spring2024,5,15,14,1,0,0.933,0.938
2,Noe Romero,Spring2023,4,11,10,2,0,0.909,0.923
3,Joab Santoyo,Spring2026,3,11,10,0,0,0.909,0.909
4,Joab Santoyo,Fall2024,5,20,17,0,0,0.850,0.850
...,...,...,...,...,...,...,...,...,...
164,David Cruz,Summer2024,8,24,7,1,0,0.292,0.320
165,David Cruz,Fall2024,6,21,6,1,0,0.286,0.318
166,David Cruz,Summer2023,8,22,6,5,0,0.273,0.407
167,Martin Cruz,Winter2024,1,1,0,0,0,0.000,0.000


In [ ]:


def player_photo_path(player_name):
    """Return the path to a player's photo, or the placeholder if missing."""
    slug = player_name.lower().replace(" ", "_")
    photo = "softball-stats/assets/" / f"{slug}.jpg"
    if photo.exists():
        return str(photo)
    # Try .png as a fallback
    photo_png = PLAYER_PHOTO_DIR / f"{slug}.png"
    if photo_png.exists():
        return str(photo_png)
    return str(PLACEHOLDER_PHOTO)

TypeError: unsupported operand type(s) for /: 'str' and 'str'